# Attention Is All You Need — 통합 실습 노트

이 노트북은 `study/attention-basics`에서 진행한 실습을 한 흐름으로 합친 학습용 정리본이다.

## 목표
1. Problem
2. Core idea
3. Method
4. Input / GT / Output / Loss
5. Evidence
6. My observation

학습용 toy 설정은 `seed=42`, `d_model=4`, `num_heads=2`, `d_ff=8`로 고정한다.

## Transformer 전체 구조

아래 그림을 먼저 보고, 이후 각 section에서 Encoder/Decoder 내부 블록을 분해해서 따라간다.

![Transformer architecture](images/Transformer_architecture.png)

> 전체 관점: **Embedding + PE → Encoder → Decoder → Linear/Softmax**

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 4
num_heads = 2
d_k = d_model // num_heads
d_v = d_model // num_heads
d_ff = 8
num_layers = 6

vocab = {
    0: "<PAD>",
    1: "<SOS>",
    2: "I",
    3: "love",
    4: "robotics",
    5: "<EOS>",
}
vocab_size = len(vocab)

print("seed =", 42)
print("d_model =", d_model)
print("num_heads =", num_heads)
print("d_k =", d_k, "d_v =", d_v)
print("d_ff =", d_ff)
print("vocab_size =", vocab_size)

## 1. Embedding + Positional Encoding

Source:
`I love robotics`

Decoder shifted target:
`<SOS> I love`

Transformer 입력은

`x = sqrt(d_model) * Embedding + PE`

로 만든다.

### 구조 그림

![Embedding and Positional Encoding](images/01_embedding_pe.png)

> 핵심: token ID를 embedding vector로 바꾼 뒤 positional encoding을 같은 차원으로 더한다.

In [ ]:
embedding = nn.Embedding(vocab_size, d_model)

with torch.no_grad():
    embedding.weight.copy_(torch.tensor([
        [0.0, 0.0, 0.0, 0.0],  # <PAD>
        [1.0, 0.0, 0.0, 1.0],  # <SOS>
        [1.0, 0.0, 1.0, 0.0],  # I
        [0.0, 1.0, 0.0, 1.0],  # love
        [1.0, 1.0, 1.0, 1.0],  # robotics
        [0.0, 0.0, 1.0, 1.0],  # <EOS>
    ], dtype=torch.float32))

def positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )

    angles = position * div_term

    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe

def embed_with_pe(token_ids):
    raw = embedding(token_ids)
    scaled = raw * math.sqrt(d_model)
    pe = positional_encoding(len(token_ids), d_model)
    return raw, scaled, pe, scaled + pe

source_ids = torch.tensor([2, 3, 4])
decoder_input_ids = torch.tensor([1, 2, 3])

source_raw, source_scaled, source_pe, x_source = embed_with_pe(source_ids)
target_raw, target_scaled, target_pe, x_decoder = embed_with_pe(decoder_input_ids)

print("Source IDs:", source_ids)
print("Source x:")
print(x_source)
print("shape:", x_source.shape)

print("\nDecoder input IDs:", decoder_input_ids)
print("Decoder x:")
print(x_decoder)
print("shape:", x_decoder.shape)

## 2. Scaled Dot-Product Attention

- Q/K: attention weight를 만드는 경로
- V: output representation을 구성하는 경로
- `QK^T`: compatibility score
- softmax: token별 mixing ratio
- `attention_weights @ V`: V들의 weighted sum

### 구조 그림

![Scaled Dot-Product Attention](images/02_attention.png)

> 핵심 흐름: **QKᵀ → scaling → softmax → V weighted sum**

In [ ]:
torch.manual_seed(42)

W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

Q = W_Q(x_source)
K = W_K(x_source)
V = W_V(x_source)

scores = Q @ K.T
scaled_scores = scores / math.sqrt(d_model)
attention_weights = F.softmax(scaled_scores, dim=-1)
attention_output = attention_weights @ V

print("W_Q.weight:\n", W_Q.weight)
print("\nW_K.weight:\n", W_K.weight)
print("\nW_V.weight:\n", W_V.weight)

print("\nQK^T:\n", scores)
print("\nAttention weights:\n", attention_weights)
print("\nAttention output:\n", attention_output)

print("\nQuery=I weights:", attention_weights[0])
print("output_I:", attention_output[0])

## 3. Multi-Head Attention

`d_model=4`, `num_heads=2`이므로 각 head는 `d_k=d_v=2`를 사용한다.

중요한 점:
- 원래 feature를 앞/뒤로 자르는 것이 아니다.
- 각 head가 전체 input을 서로 다른 learned projection으로 본다.
- 두 `[3,2]` head output을 feature dimension으로 concat하면 `[3,4]`.
- 마지막 `W_O`로 다시 `d_model` 차원 representation을 만든다.

### 구조 그림

![Multi-Head Attention](images/03_multihead.png)

> 각 head가 전체 input을 서로 다른 Q/K/V projection으로 본 뒤, head output을 concat하고 (W_O)로 다시 섞는다.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads

        self.W_Q = nn.ModuleList([
            nn.Linear(d_model, self.d_k, bias=False)
            for _ in range(num_heads)
        ])
        self.W_K = nn.ModuleList([
            nn.Linear(d_model, self.d_k, bias=False)
            for _ in range(num_heads)
        ])
        self.W_V = nn.ModuleList([
            nn.Linear(d_model, self.d_v, bias=False)
            for _ in range(num_heads)
        ])
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = Q @ K.T
        scaled_scores = scores / math.sqrt(self.d_k)

        if mask is not None:
            scaled_scores = scaled_scores.masked_fill(
                mask,
                float("-inf")
            )

        weights = F.softmax(
            scaled_scores,
            dim=-1
        )

        output = weights @ V

        return output, scores, scaled_scores, weights

    def forward(
        self,
        query_input,
        key_value_input=None,
        mask=None
    ):
        if key_value_input is None:
            key_value_input = query_input

        head_outputs = []
        head_info = []

        for i in range(self.num_heads):
            Q = self.W_Q[i](query_input)
            K = self.W_K[i](key_value_input)
            V = self.W_V[i](key_value_input)

            out, scores, scaled_scores, weights = (
                self.scaled_dot_product_attention(
                    Q, K, V, mask
                )
            )

            head_outputs.append(out)

            head_info.append({
                "Q": Q,
                "K": K,
                "V": V,
                "scores": scores,
                "scaled_scores": scaled_scores,
                "weights": weights,
                "output": out,
            })

        concat = torch.cat(
            head_outputs,
            dim=-1
        )

        output = self.W_O(concat)

        return output, concat, head_info

torch.manual_seed(42)

mha = MultiHeadAttention(
    d_model,
    num_heads
)

mha_output, concat, mha_info = mha(x_source)

for h, info in enumerate(mha_info, start=1):
    print(f"\n=== Head {h} ===")
    print("Attention weights:")
    print(info["weights"])
    print("Head output:")
    print(info["output"])

print("\nConcat:")
print(concat, concat.shape)

print("\nW_O:")
print(mha.W_O.weight)

print("\nMHA output:")
print(mha_output, mha_output.shape)

## 4. Encoder

한 Encoder Layer:

`Self-Attention -> Residual + LayerNorm -> FFN -> Residual + LayerNorm`

- Attention: token 간 정보 교환
- FFN: 각 token 내부 feature 변환
- Residual: 기존 representation 보존 + gradient flow 도움
- LayerNorm: token 내부 feature scale 안정화

Encoder Layer를 6번 반복하지만 parameter는 공유하지 않는다.

### Encoder 구조 그림

![Transformer Encoder](images/04_encoder.png)

> 한 Encoder Layer는 **Self-Attention → Add & Norm → FFN → Add & Norm** 순서이며, 동일한 구조를 여러 layer에 반복하지만 parameter는 공유하지 않는다.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attention_output, _, head_info = self.attention(x)

        residual1 = x + attention_output
        x1 = self.norm1(residual1)

        ffn_output = self.ffn(x1)

        residual2 = x1 + ffn_output
        output = self.norm2(residual2)

        return output, {
            "attention_output": attention_output,
            "residual1": residual1,
            "norm1_output": x1,
            "ffn_output": ffn_output,
            "residual2": residual2,
            "output": output,
            "head_info": head_info,
        }

class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):
        all_results = []

        for layer in self.layers:
            x, result = layer(x)
            all_results.append(result)

        return x, all_results

torch.manual_seed(42)

encoder = Encoder(
    d_model,
    num_heads,
    d_ff,
    num_layers
)

encoder_output, encoder_results = encoder(x_source)

r1 = encoder_results[0]

print("Encoder Layer 1")
print("\nAttention output:")
print(r1["attention_output"])

print("\nResidual 1:")
print(r1["residual1"])

print("\nAfter LayerNorm 1:")
print(r1["norm1_output"])

print("\nFFN output:")
print(r1["ffn_output"])

print("\nLayer output:")
print(r1["output"])

print("\nLayerNorm statistics")
print("before mean:", r1["residual1"].mean(dim=-1))
print("before std :", r1["residual1"].std(dim=-1, unbiased=False))
print("after mean :", r1["norm1_output"].mean(dim=-1))
print("after std  :", r1["norm1_output"].std(dim=-1, unbiased=False))

print("\nFinal encoder output:")
print(encoder_output)
print("shape:", encoder_output.shape)

## 5. Decoder — Outputs (shifted right)

Target GT:
`I love robotics`

Decoder input:
`<SOS> I love`

즉:
- `<SOS>` -> `I`
- `<SOS> I` -> `love`
- `<SOS> I love` -> `robotics`

Training에서는 shifted target을 한 번에 넣되 causal mask로 미래 token을 보지 못하게 한다.

Cross-Attention:
- Q <- Decoder
- K/V <- Encoder output을 다시 projection

### Decoder 구조 그림

![Transformer Decoder](images/05_decoder.png)

> Decoder에서는 **Masked Self-Attention**과 **Encoder-Decoder Cross-Attention**이 추가된다. Causal mask 때문에 미래 target token은 볼 수 없다.

In [ ]:
target_len = x_decoder.shape[0]

causal_mask = torch.triu(
    torch.ones(
        target_len,
        target_len,
        dtype=torch.bool
    ),
    diagonal=1
)

print("Causal mask:")
print(causal_mask)

class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff
    ):
        super().__init__()

        self.masked_self_attention = MultiHeadAttention(
            d_model,
            num_heads
        )
        self.norm1 = nn.LayerNorm(d_model)

        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        x,
        encoder_output,
        causal_mask
    ):
        masked_out, _, masked_info = (
            self.masked_self_attention(
                query_input=x,
                key_value_input=x,
                mask=causal_mask
            )
        )

        x1 = self.norm1(
            x + masked_out
        )

        cross_out, _, cross_info = (
            self.cross_attention(
                query_input=x1,
                key_value_input=encoder_output,
                mask=None
            )
        )

        x2 = self.norm2(
            x1 + cross_out
        )

        ffn_out = self.ffn(x2)

        output = self.norm3(
            x2 + ffn_out
        )

        return output, {
            "masked_info": masked_info,
            "cross_info": cross_info,
            "output": output,
        }

class Decoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model,
                num_heads,
                d_ff
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        x,
        encoder_output,
        causal_mask
    ):
        all_results = []

        for layer in self.layers:
            x, result = layer(
                x,
                encoder_output,
                causal_mask
            )
            all_results.append(result)

        return x, all_results

torch.manual_seed(42)

decoder = Decoder(
    d_model,
    num_heads,
    d_ff,
    num_layers
)

decoder_output, decoder_results = decoder(
    x_decoder,
    encoder_output,
    causal_mask
)

d1 = decoder_results[0]

print("\nDecoder Layer 1 Head 1 masked attention weights:")
print(d1["masked_info"][0]["weights"])

print("\nDecoder Layer 1 Head 1 cross-attention weights:")
print(d1["cross_info"][0]["weights"])

print("\nFinal decoder output:")
print(decoder_output)
print("shape:", decoder_output.shape)

## 6. Output / GT / Loss

`decoder_output [target_len, d_model]`

-> Output Linear

-> `logits [target_len, vocab_size]`

GT:
`I love robotics -> [2,3,4]`

각 position의 Cross Entropy는
`-log P(correct token)`이다.

PyTorch `CrossEntropyLoss`에는 raw logits를 넣는다.

In [ ]:
torch.manual_seed(42)

output_linear = nn.Linear(
    d_model,
    vocab_size,
    bias=False
)

logits = output_linear(
    decoder_output
)

probabilities = F.softmax(
    logits,
    dim=-1
)

pred_ids = torch.argmax(
    probabilities,
    dim=-1
)

gt_ids = torch.tensor([
    2, 3, 4
])

gt_probabilities = probabilities[
    torch.arange(len(gt_ids)),
    gt_ids
]

manual_losses = -torch.log(
    gt_probabilities
)

manual_loss = manual_losses.mean()

criterion = nn.CrossEntropyLoss()

ce_loss = criterion(
    logits,
    gt_ids
)

criterion_smoothing = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

smoothed_loss = criterion_smoothing(
    logits,
    gt_ids
)

print("Logits:")
print(logits)

print("\nProbabilities:")
print(probabilities)

print("\nPredicted IDs:", pred_ids)
print(
    "Predicted tokens:",
    [vocab[i.item()] for i in pred_ids]
)

print("\nGT IDs:", gt_ids)
print(
    "GT tokens:",
    [vocab[i.item()] for i in gt_ids]
)

print("\nGT probabilities:", gt_probabilities)
print("Manual CE per position:", manual_losses)
print("Manual CE mean:", manual_loss)
print("PyTorch CE:", ce_loss)
print("Label smoothing CE:", smoothed_loss)

## 7. 전체 학습 흐름

```text
Source
  ↓
Token IDs
  ↓
Embedding + PE
  ↓
Encoder × 6
  ↓
encoder_output
        \
         → Cross-Attention
        /
Shifted Target
  ↓
Embedding + PE
  ↓
Masked Self-Attention
  ↓
Cross-Attention
  ↓
FFN
  ↓
Decoder × 6
  ↓
decoder_output
  ↓
Linear
  ↓
logits
  ↓
CrossEntropy with GT
  ↓
backpropagation
```

## 8. 논문 정리

### Problem
RNN 계열은 순차 계산 때문에 병렬화가 어렵고 long-range dependency의 경로가 길다.
CNN은 병렬화는 가능하지만 distant position을 연결하려면 여러 layer가 필요하다.

### Core idea
Recurrence와 convolution을 제거하고 token 관계를 Self-Attention으로 직접 계산한다.

### Method
Embedding + PE -> Attention -> Multi-Head -> Encoder -> Decoder -> Output Linear

### Input / GT / Output / Loss
- Encoder input: source sequence
- Decoder input: target sequence shifted right
- GT: 원래 target sequence
- Output: vocabulary logits / next-token probability
- Loss: Cross Entropy + label smoothing

### Evidence
- Self-Attention의 sequential operations와 path length 장점
- WMT14 번역 성능
- head 수, d_k, model size, dropout, positional encoding variation 실험

### My observation
- Q/K는 attention weight를 만든다.
- V는 weighted sum되어 output representation을 만든다.
- 각 head는 전체 input을 서로 다른 projection으로 본다.
- Concat은 feature dimension으로 이어 붙인다.
- Residual 때문에 sub-layer output shape는 d_model과 맞아야 한다.
- LayerNorm 후 token별 mean≈0, std≈1을 확인할 수 있다.
- Causal mask를 적용하면 미래 token attention weight가 0이 된다.
- random 초기화 상태의 prediction 자체에는 의미가 없다.

### 가져갈 한 문장
**Transformer는 recurrence 없이 attention으로 token 간 관계를 직접 계산하고,
그 representation을 반복적으로 가공해 next-token prediction에 사용한다.**